# Transformer Fine-tuning Experiments
**DistilBERT + BERT + RoBERTa × 7 source configs = 21 experiments**

Requires GPU runtime. Estimated time: ~6-8 hours on T4.

Place the `data/splits/` folder under `/content/data/splits`. If Colab disconnects, re-run the notebook — completed experiments are skipped automatically.

## 1. Setup

In [10]:
!pip install -q transformers

# === Path Configuration ===
# Place the data/splits folder under /content/data/splits
SPLITS_DIR = "/content/data/splits"
RESULTS_DIR = "/content/results"
MODELS_DIR = "/content/models"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

files = os.listdir(SPLITS_DIR)
print(f"Found {len(files)} split files")
assert len(files) >= 21

Found 22 split files


In [15]:
import csv
import random
from collections import Counter
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

# ====== Config ======
SEED = 42
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_NAMES = ["negative", "neutral", "positive"]

TRANSFORMER_MODELS = {
    "distilbert": "distilbert-base-uncased",
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
}
TRANSFORMER_MAX_LENGTH_TWEET = 128
TRANSFORMER_MAX_LENGTH_REVIEW = 512
TRANSFORMER_EPOCHS = 4
TRANSFORMER_BATCH_SIZE = 16
TRANSFORMER_LEARNING_RATE = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01

ALL_CONFIGS = [
    "twitter", "skytrax", "airlinequality",
    "twitter+skytrax", "twitter+airlinequality", "skytrax+airlinequality",
    "twitter+skytrax+airlinequality"
]

LOG_FILE = os.path.join(RESULTS_DIR, "experiment_log_transformers.csv")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 2. Utilities

In [16]:
def load_split(source_config, split):
    path = os.path.join(SPLITS_DIR, f"{source_config}_{split}.csv")
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(LABEL_MAP[row["label"]])
    return texts, labels


def get_max_length(source_config):
    return TRANSFORMER_MAX_LENGTH_TWEET if source_config == "twitter" else TRANSFORMER_MAX_LENGTH_REVIEW


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro"),
        "macro_recall": recall_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def compute_per_class_metrics(y_true, y_pred):
    result = {}
    for avg in ["macro", "weighted", None]:
        p = precision_score(y_true, y_pred, average=avg, zero_division=0)
        r = recall_score(y_true, y_pred, average=avg, zero_division=0)
        f = f1_score(y_true, y_pred, average=avg, zero_division=0)
        if avg is None:
            for i, name in enumerate(LABEL_NAMES):
                result[f"{name}_precision"] = p[i]
                result[f"{name}_recall"] = r[i]
                result[f"{name}_f1"] = f[i]
        else:
            result[f"{avg}_precision"] = p
            result[f"{avg}_recall"] = r
            result[f"{avg}_f1"] = f
    return result


def get_completed_experiments():
    """Load already-completed experiments from log (for crash recovery)."""
    completed = set()
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                completed.add((row["model"], row["source_config"]))
    return completed

print("Utilities loaded.")

Utilities loaded.


## 3. Dataset & Training

In [17]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += outputs.loss.item() * len(labels)
        correct += (outputs.logits.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


def evaluate_model(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item() * len(labels)
            all_preds.extend(outputs.logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(all_labels), np.array(all_preds), np.array(all_labels)

print("Training functions defined.")

Training functions defined.


In [18]:
def run_experiment(model_key, source_config):
    """Fine-tune and evaluate one transformer on one source config."""
    set_seed()
    model_name = TRANSFORMER_MODELS[model_key]
    max_length = get_max_length(source_config)

    print(f"\n{'='*60}")
    print(f"[TRAIN] {model_key.upper()} ({model_name}) | {source_config}")
    print(f"  max_length={max_length}, epochs={TRANSFORMER_EPOCHS}, batch={TRANSFORMER_BATCH_SIZE}")
    print(f"{'='*60}")

    # Load data
    train_texts, train_labels = load_split(source_config, "train")
    val_texts, val_labels = load_split(source_config, "val")
    test_texts, test_labels = load_split(source_config, "test")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = TextDataset(train_texts, train_labels, tokenizer, max_length)
    val_ds = TextDataset(val_texts, val_labels, tokenizer, max_length)
    test_ds = TextDataset(test_texts, test_labels, tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=TRANSFORMER_BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=TRANSFORMER_BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LEARNING_RATE,
                                  weight_decay=TRANSFORMER_WEIGHT_DECAY)
    total_steps = len(train_loader) * TRANSFORMER_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

    best_val_f1 = 0
    best_state = None

    for epoch in range(1, TRANSFORMER_EPOCHS + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler)
        val_loss, val_preds, val_labels_arr = evaluate_model(model, val_loader)
        val_f1 = f1_score(val_labels_arr, val_preds, average="macro")

        print(f"  Epoch {epoch}/{TRANSFORMER_EPOCHS} | "
              f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restore best and evaluate on test
    model.load_state_dict(best_state)
    model = model.to(device)
    _, test_preds, test_labels_arr = evaluate_model(model, test_loader)

    metrics = compute_metrics(test_labels_arr, test_preds)
    per_class = compute_per_class_metrics(test_labels_arr, test_preds)
    cm = confusion_matrix(test_labels_arr, test_preds)

    print(f"  TEST | Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
    print(f"  Confusion Matrix:\n{cm}")

    # Save model to Drive
    save_dir = os.path.join(MODELS_DIR, model_key, source_config)
    os.makedirs(save_dir, exist_ok=True)
    model.cpu().save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"  Model saved to {save_dir}")

    # Free GPU memory
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

    return {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": model_key,
        "source_config": source_config,
        **metrics, **per_class,
    }

print("Experiment function defined.")

Experiment function defined.


## 4. Run All 21 Experiments

Completed experiments are automatically skipped on re-run (crash recovery).

In [19]:
# Load any previously completed experiments
completed = get_completed_experiments()
all_results = []
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, encoding="utf-8") as f:
        all_results = list(csv.DictReader(f))

print(f"Already completed: {len(completed)} experiments")

total_experiments = len(TRANSFORMER_MODELS) * len(ALL_CONFIGS)
exp_num = len(completed)

for model_key in TRANSFORMER_MODELS:
    for source_config in ALL_CONFIGS:
        if (model_key, source_config) in completed:
            print(f"[SKIP] {model_key} | {source_config} (already done)")
            continue

        result = run_experiment(model_key, source_config)
        all_results.append(result)
        exp_num += 1

        # Save after each experiment (crash recovery)
        with open(LOG_FILE, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
            writer.writeheader()
            writer.writerows(all_results)
        print(f"  [{exp_num}/{total_experiments}] Progress saved.")

print(f"\n{'='*60}")
print(f"DONE: All {total_experiments} experiments completed.")

Already completed: 0 experiments

[TRAIN] DISTILBERT (distilbert-base-uncased) | twitter
  max_length=128, epochs=4, batch=16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/4 | train_loss=0.5283 acc=0.7996 | val_loss=0.4299 val_f1=0.7437
  Epoch 2/4 | train_loss=0.4031 acc=0.8501 | val_loss=0.4430 val_f1=0.7322
  Epoch 3/4 | train_loss=0.3225 acc=0.8840 | val_loss=0.4743 val_f1=0.7466
  Epoch 4/4 | train_loss=0.2525 acc=0.9140 | val_loss=0.5206 val_f1=0.7566
  TEST | Accuracy: 0.8362 | Macro-F1: 0.7507
  Confusion Matrix:
[[4956  322  372]
 [ 378  620  310]
 [ 162  255 3605]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model saved to /content/models/roberta/twitter+skytrax+airlinequality
  [21/21] Progress saved.

DONE: All 21 experiments completed.


## 5. Results Summary

In [20]:
# Reload from file to get clean data
with open(LOG_FILE, encoding="utf-8") as f:
    final_results = list(csv.DictReader(f))

print(f"{'Model':<12} {'Source Config':<35} {'Accuracy':>10} {'Macro-F1':>10}")
print("=" * 72)
for r in final_results:
    print(f"{r['model']:<12} {r['source_config']:<35} {float(r['accuracy']):>10.4f} {float(r['macro_f1']):>10.4f}")

best = max(final_results, key=lambda x: float(x["macro_f1"]))
print(f"\nBest: {best['model']} | {best['source_config']} | Macro-F1: {float(best['macro_f1']):.4f}")

Model        Source Config                         Accuracy   Macro-F1
distilbert   twitter                                 0.8333     0.7847
distilbert   skytrax                                 0.8183     0.6942
distilbert   airlinequality                             0.8264     0.6340
distilbert   twitter+skytrax                         0.8296     0.7495
distilbert   twitter+airlinequality                     0.8295     0.7303
distilbert   skytrax+airlinequality                     0.8294     0.7210
distilbert   twitter+skytrax+airlinequality             0.8266     0.7374
bert         twitter                                 0.8310     0.7730
bert         skytrax                                 0.8270     0.7111
bert         airlinequality                             0.8243     0.6616
bert         twitter+skytrax                         0.8305     0.7494
bert         twitter+airlinequality                     0.8240     0.7265
bert         skytrax+airlinequality                     0.8

## 6. Download Results

In [22]:
import subprocess
#subprocess.run(["zip", "-r", "/content/results_all.zip", "/content/results", "/content/models"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_skytrax.zip", "/content/models/bert/skytrax"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_skytrax_airlinequality.zip", "/content/models/bert/skytrax+airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_airlinequality.zip", "/content/models/bert/airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_twitter.zip", "/content/models/bert/twitter"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_twitter_skytrax.zip", "/content/models/bert/twitter+skytrax"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_twitter_skytrax_airlinequality.zip", "/content/models/bert/twitter+skytrax+airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_bert_twitter_airlinequality.zip", "/content/models/bert/twitter+airlinequality"], check=True)

subprocess.run(["zip", "-r", "/content/models_distilbert_skytrax.zip", "/content/models/distilbert/skytrax"], check=True)
subprocess.run(["zip", "-r", "/content/models_distilbert_skytrax_airlinequality.zip", "/content/models/distilbert/skytrax+airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_distilbert_airlinequality.zip", "/content/models/distilbert/airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_distilbert_twitter.zip", "/content/models/distilbert/twitter"], check=True)
subprocess.run(["zip", "-r", "/content/models_distilbert_twitter_skytrax.zip", "/content/models/distilbert/twitter+skytrax"], check=True)
subprocess.run(["zip", "-r", "/content/models_distilbert_twitter_skytrax_airlinequality.zip", "/content/models/distilbert/twitter+skytrax+airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_distilbert_twitter_airlinequality.zip", "/content/models/bert/twitter+airlinequality"], check=True)

subprocess.run(["zip", "-r", "/content/models_roberta_skytrax.zip", "/content/models/roberta/skytrax"], check=True)
subprocess.run(["zip", "-r", "/content/models_roberta_skytrax_airlinequality.zip", "/content/models/roberta/skytrax+airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_roberta_airlinequality.zip", "/content/models/roberta/airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_roberta_twitter.zip", "/content/models/roberta/twitter"], check=True)
subprocess.run(["zip", "-r", "/content/models_roberta_twitter_skytrax.zip", "/content/models/roberta/twitter+skytrax"], check=True)
subprocess.run(["zip", "-r", "/content/models_roberta_twitter_skytrax_airlinequality.zip", "/content/models/roberta/twitter+skytrax+airlinequality"], check=True)
subprocess.run(["zip", "-r", "/content/models_roberta_twitter_airlinequality.zip", "/content/models/roberta/twitter+airlinequality"], check=True)

print("Zip created: /content/results_all.zip")
print("This contains all results and models. Download from the file explorer.")

Zip created: /content/results_all.zip
This contains all results and models. Download from the file explorer.


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
